# Phase 13 — Final Held-Out Ablation Evaluation

**Preparation only until explicit user confirmation.** MorphoNet-BT: Privileged Morphological Supervision for Lightweight Brain Tumor Classification in Contrast-Enhanced MRI.

## Public-copy execution policy
This is an English rendering of a historical research notebook. Outputs and execution counts were cleared; no cell was rerun for Phase 18. Historical findings below describe earlier runs only. Every code cell is gated, including setup cells that may perform model checks, preprocessing or writes. Enable `CONFIRM_PUBLIC_WORKFLOW` deliberately in the first code cell only after reviewing [the reproduction guide](../docs/reproducibility.md). Existing stage-specific confirmations must also be enabled separately. Run from the repository root or set `MORPHONET_ROOT` to a private writable experiment workspace with the same source/config layout; set `MORPHONET_DATA_ROOT` for dataset discovery. Original source/configuration fingerprints, private split manifests and checkpoints are not portable artifacts; do not bypass their validation. See the guide for prerequisites and limitations.


## Scientific objective and frozen protocol

Quantify the cumulative contributions of MSS, privileged morphology supervision, BAA, global/local dual path and AMGF. A0–A4 use 25 validation-selected best checkpoints; A5 uses the frozen Phase 7 predictions. Sequential differences are conditional on order and do not establish independent effects, interactions or statistical significance. No model, checkpoint selection, preprocessing, split or training setting may change after Test access.

## Mandatory manual execution policy

The agent prepares this notebook without decoding Test images, running Test inference or calculating new A0–A4 held-out metrics. `CONFIRM_TEST_EVALUATION` remains `False`. Only the user may set it to `True` in cell `MANUAL-13` and execute that cell. The safe preparation runner physically excludes that cell. No partial fold, background job or automatic evaluation is permitted.

## Imports and environment

In [ ]:
CONFIRM_PUBLIC_WORKFLOW = False  # Explicit opt-in; no cell has been rerun.
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
import os, sys
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
ROOT = Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0,str(ROOT))
import pandas as pd
from IPython.display import display, Image
from src.evaluation.ablation_validation import safe_prepare
from src.evaluation import ablation_protocol
pd.set_option('display.max_columns', 15)
pd.set_option('display.precision', 5)
OUT = ROOT / 'results/ablation/evaluation'

## Frozen artifact discovery and checkpoint integrity

Locate all 25 completed runs and load only `best_macro_f1.pt` on CPU. Inspect the actual Phase 12 schema and verify variant/fold, architecture, class mapping, preprocessing, split/config/source fingerprints, best epoch, validation selection, completion hashes and strict model-state loading. No optimizer state is applied. All 25 must pass before evaluation is enabled. Missing or inconsistent artifacts block evaluation; no substitutions are made.

## Frozen split and protected-artifact audit

Read CSV identities only to verify all five patient-separated folds. Expected slice counts: 613, 614, 611, 613, 613; patient counts: 47, 47, 46, 47, 47. Dataset total: 3064 slices and 233 patients. Existing main, baseline, Phase 11 and Phase 12 artifacts are protected with hashes; last checkpoints may be hashed as opaque files but are never deserialized for evaluation.

## Non-Test preparation checks

The following cell blocks Dataset construction, model forward calls and optimizer steps while verifying checkpoints. Metric, probability aggregation, alignment and bootstrap tests use synthetic records only. Frozen A5 predictions may be read for reference verification. No new held-out results are produced.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
context, preparation_audit = safe_prepare(ROOT)
display(context['checkpoints'][['Variant','Fold','BestEpoch','BestValidationMacroF1','TestPatients','TestSlices']])
display(pd.Series(preparation_audit,name='Preparation audit'))

## Evaluation configuration and model factory

Use unchanged Phase 11 factories and frozen complexity records. MRI-only FP32 inference uses batch 16, zero workers, deterministic algorithms, disabled TF32 and seed 42, matching Phase 7. The parameter table includes training-only auxiliary heads; inference-required parameter inventory is reported separately after manual execution without modifying models or recomputing MACs.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(context['complexity'][['MSS','Morphology Supervision','BAA','Dual Path','AMGF','Parameters','Inference MACs','Inference FLOPs']])

## Deterministic Test dataset and MRI-only inference

Test Dataset construction occurs only behind the manual guard. Use frozen grayscale MRI, bilinear resize to 256×256 and per-image Z-score without clipping. Test mode disables affine, gain, noise and flips. Set `include_targets=False`. The batch must contain only image, label and identifiers. The model receives only `model(images)` in eval/inference mode and returns `[B,3]`; probabilities come from softmax. BAA guidance is predicted from MRI features, never GT targets.

## Resume, idempotency and overwrite protection

Each completed fold has a receipt binding checkpoint, variant/fold, Test split, preprocessing, class order, evaluation source/configuration and software versions to the prediction hash. With resume enabled, exact matching results are reused without inference. All selected caches are checked before the first image is decoded.

An incomplete started fold or any mismatch stops for explicit recovery review. `OVERWRITE=True` is deliberately rejected; it never authorizes repeated evaluation or silently replaces predictions. Completed folds remain reusable after an interruption. Do not delete receipts to force another pass.

## Metric utilities

Per-fold and pooled OOF metrics include accuracy, macro precision/recall/F1, weighted F1 and macro OVR ROC-AUC. Per-class metrics include precision, sensitivity, specificity, F1, AUC and support. Undefined denominators or absent-class AUC produce explicit NaN values; macro summaries propagate them rather than silently substituting zero. CSV blank fields represent undefined values. ROC uses probabilities, not hard labels.

## OOF and patient aggregation

All five folds must provide exactly 3064 unique samples and 233 patients, with exact identities, labels, class support and frozen fold assignments. Each patient belongs to one outer Test fold. All six coverage audits must pass before final comparison tables are created. OOF metrics are recomputed from concatenated predictions; five-fold mean and sample SD (ddof=1) are separate. Patient predictions average slice probabilities in float64 and take argmax, producing 233 predictions.

## Frozen A5 reference loader

Read original complete OOF and patient prediction files and their fold receipts. Check exact identities, labels and assignments. Phase 7 OOF CSV reserialization introduces only sub-femtoscale probability differences from its fold files; an absolute tolerance of 3e-16 is allowed for those probability comparisons only. Identity and class checks remain exact. A5 is never inferred again or retrained; derived reports are written under `A5_reference/`, without copying checkpoints.

## MANUAL HELD-OUT TEST EVALUATION — USER ONLY

**Cell `MANUAL-13` is the single evaluation launch cell. It is delivered unexecuted.**

After reviewing preparation, manually change only `CONFIRM_TEST_EVALUATION` to `True` and run the cell. The default selection authorizes A0 Fold 1–5, then A1–A4 in order, followed by complete reporting. The function guard runs before preflight or any Test dataset access. Subsequent notebook cells display the returned results without repeating inference.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
# MANUAL-13 — USER EXECUTION ONLY. Do not execute during preparation.
VARIANTS_TO_EVALUATE = ["A0", "A1", "A2", "A3", "A4"]
FOLDS_TO_EVALUATE = [1, 2, 3, 4, 5]
CONFIRM_TEST_EVALUATION = False
OVERWRITE = False
RESUME_EVALUATION = True

if not CONFIRM_TEST_EVALUATION:
    raise RuntimeError(
        "Held-out Test evaluation is disabled. "
        "The user must manually set CONFIRM_TEST_EVALUATION = False."
    )

import importlib
ablation_protocol = importlib.reload(ablation_protocol)
tables = ablation_protocol.launch(
    ROOT, variants=VARIANTS_TO_EVALUATE, folds=FOLDS_TO_EVALUATE,
    confirm_test_evaluation=CONFIRM_TEST_EVALUATION,
    overwrite=OVERWRITE, resume_evaluation=RESUME_EVALUATION,
)
if tables is not None:
    display(tables["final_ablation_comparison"])
    display(tables["patient_level_ablation_comparison"])


## Per-fold results

All five folds, including frozen Phase 7 A5 Test results. Validation results are never substituted.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['per_fold_comparison'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Primary OOF comparison

All six variants with component flags, parameters and inference MACs. Best actual values are identified programmatically; A5 is not assumed to lead.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['final_ablation_comparison'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Expanded OOF metrics

Full six-metric supplementary comparison.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['expanded_ablation_comparison'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Five-fold mean and sample SD

Separate from pooled OOF metrics; ddof=1.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['five_fold_mean_sd'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Patient-level results

Mean softmax probability per patient.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['patient_level_ablation_comparison'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Per-class slice results

Fixed class ordering: Meningioma, Glioma, Pituitary.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['per_class_metrics'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Per-class patient results

Patient-level support and class metrics.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['patient_per_class_metrics'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Sequential component contributions

Signed later-minus-earlier differences in percentage points. Negative contributions remain visible; monotonic improvement is not assumed.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['sequential_component_contributions'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Fold-level changes

Identify increases and decreases descriptively without changing experiments.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['per_fold_sequential_differences'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Meningioma analysis

Precision, sensitivity, specificity, F1 and AUC for each variant.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['meningioma_comparison'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Meningioma sensitivity by fold

Class-specific variation across the same frozen folds.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['meningioma_by_fold'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Fold 2 analysis

Descriptive accuracy, macro F1 and all three class recalls; no tuning.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['fold_2_comparison'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Error counts

All misclassified slices and patients retained in detailed files; per-class errors and confusion patterns are also saved.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['misclassification_counts'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Paired error overlap

Consecutive pairs, separately at slice and patient levels; both correct, earlier only, later only and both wrong.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['paired_error_overlap'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Patient-cluster bootstrap

2,000 shared patient draws, seed 42, percentile 95% CI. All slices and repeated-cluster multiplicities are retained. Intervals condition on fixed trained models/predictions and exclude retraining uncertainty.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['accuracy_cluster_bootstrap_ci'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Accuracy and efficiency

Frozen total parameters and inference MACs remain unchanged. A2 auxiliary-head parameters add training capacity without increasing primary inference MACs; inference-required parameter bytes exclude training-only heads and do not describe total runtime memory.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(tables['accuracy_efficiency'])
else:
    print('Not evaluated: awaiting manual user execution.')

## Confusion matrices and ROC curves

For A0–A5, generate raw OOF slice, row-normalized slice and raw patient confusion matrices (CSV/PNG/PDF). Rows are true and columns predicted classes. OVR ROC curves use each class probability. Figures are generated only after user execution.

## Statistical preparation

Aligned slice and patient files contain identities, true labels/folds, each variant prediction and all probabilities. Exact alignment is mandatory. Shared bootstrap indices, PID ordering and replicates are retained. No statistical significance testing is performed; interval overlap or non-overlap alone does not establish significance.

## Publication figures

Generate eight summary charts: OOF macro F1, OOF accuracy, patient macro F1, signed sequential F1 differences, parameters, MACs, Meningioma sensitivity and fold macro F1. Save 400-dpi PNG and vector PDF with separate units and zero-based axes; signed differences straddle zero.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
if 'tables' in globals() and tables is not None:
    display(Image(filename=str(OUT / 'figures/OOF_Macro_F1.png'), width=650))
    display(Image(filename=str(OUT / 'figures/sequential_macro_f1.png'), width=650))
else:
    print('Publication figures will be generated after manual evaluation.')

## Final scientific audit

After the user completes evaluation, `final_scientific_audit.json` records 25 best evaluations or verified cache reuse, zero last-checkpoint inference, zero training/optimizer steps, MRI-only deterministic inputs, exact OOF coverage, patient and per-class metrics, shared patient bootstrap, signed differences, paired alignment and protected-artifact preservation. Preparation writes a separate audit and does not claim evaluation completion.

## Conclusions and interpretation

Observed numerical differences will be reported after manual execution. Supported conclusions are conditional cumulative differences under this fixed protocol; independent effects and interactions remain unresolved. Statistical reliability and external-institution generalization require separate evidence. Do not claim every component helps, clinical validation or statistical significance. If A5 is not best or a component reduces performance, preserve and report that result. No retraining, redesign, patient removal or Phase 14 work follows automatically.

**Prepared only. Wait for the user to enable and execute MANUAL-13.**